In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import json
import warnings
warnings.filterwarnings('ignore')

try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

print("="*80)
print("MULTI-STATION CNN-LSTM MODEL - TRAINING WITH OPTIMAL PARAMETERS")
print("="*80)

MULTI-STATION CNN-LSTM MODEL - TRAINING WITH OPTIMAL PARAMETERS


# 1. LOAD BEST PARAMETERS

In [2]:
print("\n[1/7] Loading best parameters from grid search...")

try:
    with open('multistation_cnn_lstm_best_params.json', 'r') as f:
        best_params = json.load(f)
    print("[OK] Best parameters loaded")
except FileNotFoundError:
    print("[WARNING] No best_params.json found, using defaults")
    best_params = {
        'filters': 128,
        'dropout': 0.3,
        'lstm_units': 64,
        'learning_rate': 0.001,
        'batch_size': 32,
        'dense_units': 128,
        'n_in': 24
    }

print("\nBest Parameters:")
for param, value in best_params.items():
    print(f"  {param}: {value}")



[1/7] Loading best parameters from grid search...
[WARNING] No best_params.json found, using defaults

Best Parameters:
  filters: 128
  dropout: 0.3
  lstm_units: 64
  learning_rate: 0.001
  batch_size: 32
  dense_units: 128
  n_in: 24


# 2. LOAD AND PREPARE DATA

In [3]:
print("\n[2/7] Loading data...")

em = pd.read_csv('MiddelburgIM.csv', sep=';', header=0, index_col=0)
mb = pd.read_csv('eMalahleniIM.csv', sep=';', header=0, index_col=0)

min_len = min(len(em), len(mb))
em = em[:min_len]
mb = mb[:min_len]

target = em['pm2.5'].values.reshape(-1, 1)

# Extract ALL features INCLUDING PM2.5 from both stations (matches LSTM approach)
# This way we have PM2.5(t-1) as a feature, predicting PM2.5(t)
em_all = em.values  # All 13 features including PM2.5
mb_all = mb.values  # All 13 features including PM2.5

# Combine features from both stations: [13 + 13 = 26 features]
combined_features = np.concatenate([em_all, mb_all], axis=1)

scaler_features = MinMaxScaler()
scaler_target = MinMaxScaler()

combined_norm = scaler_features.fit_transform(combined_features)
target_norm = scaler_target.fit_transform(target)

print(f"Data loaded: Combined features {combined_norm.shape}, Target {target_norm.shape}")


[2/7] Loading data...
Data loaded: Combined features (87646, 26), Target (87646, 1)


# 3. CREATE SEQUENCES

In [4]:
print("\n[3/7] Creating sequences...")

def create_sequences(data, target_data, n_in=1):
    X, y = [], []
    
    for i in range(len(data) - n_in):
        X.append(data[i:i+n_in])
        y.append(target_data[i+n_in])
    
    return np.array(X), np.array(y)

X, y = create_sequences(combined_norm, target_norm,
                        n_in=best_params['n_in'])

print(f"Sequences created: X {X.shape}, y {y.shape}")
print(f"  Features: {X.shape[1]} (26 features × 24 timesteps)")


[3/7] Creating sequences...
Sequences created: X (87622, 24, 26), y (87622, 1)
  Features: 24 (26 features × 24 timesteps)


# 4. SPLIT DATA

In [5]:
print("\n[4/7] Splitting data...")

# Use deterministic splitting with random_state=42 for reproducibility
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.20, random_state=42)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")


[4/7] Splitting data...
Train: (56077, 24, 26), Val: (14020, 24, 26), Test: (17525, 24, 26)


# 5. BUILD AND TRAIN MODEL

In [6]:
print("\n[5/7] Building and training model...")

keras.backend.clear_session()

model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1], X_train.shape[2])),
    
    # Conv1D layers with SpatialDropout1D
    layers.Conv1D(best_params['filters'], 3, padding='same', activation='relu'),
    #layers.BatchNormalization(),
    #layers.SpatialDropout1D(best_params['dropout']),
    
    #layers.Conv1D(best_params['filters'], 3, padding='same', activation='relu'),
    #layers.BatchNormalization(),
    #layers.SpatialDropout1D(best_params['dropout']),
    
    # LSTM layer
    layers.LSTM(best_params['lstm_units'], return_sequences=True),
    #layers.Dropout(best_params['dropout']),
    layers.LSTM(best_params['lstm_units'], return_sequences=False),
    #layers.Dropout(best_params['dropout']),
    #layers.LayerNormalization(), 
    
    # Dense layers
    layers.Dense(best_params['dense_units'], activation='relu'),
    layers.Dropout(best_params['dropout']),
    
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(1)
])

optimizer = keras.optimizers.Adam(
    learning_rate=best_params['learning_rate']
)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae'])

print("\nModel Summary:")
print("-" * 80)
model.summary()

callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-6,
        verbose=1
    )
]

print(f"\nTraining with {len(X_train)} samples...")
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=best_params['batch_size'],
    callbacks=callbacks,
    verbose=1
)

print(f"\nTraining completed in {len(history.history['loss'])} epochs")


[5/7] Building and training model...


Model Summary:
--------------------------------------------------------------------------------


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 24, 128)        │        10,112 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 24, 64)         │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │         8,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         4,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 105,025 (410.25 KB)

 Trainable params: 105,025 (410.25 KB)

 Non-trainable params: 0 (0.00 B)


Training with 56077 samples...
Epoch 1/100
1753/1753 ━━━━━━━━━━━━━━━━━━━━ 70s 37ms/step - loss: 0.0022 - mae: 0.0294 - val_loss: 9.5006e-04 - val_mae: 0.0188 - learning_rate: 0.0010
Epoch 2/100
1753/1753 ━━━━━━━━━━━━━━━━━━━━ 63s 36ms/step - loss: 9.8994e-04 - mae: 0.0186 - val_loss: 9.3347e-04 - val_mae: 0.0186 - learning_rate: 0.0010
Epoch 3/100
1753/1753 ━━━━━━━━━━━━━━━━━━━━ 64s 36ms/step - loss: 9.3457e-04 - mae: 0.0179 - val_loss: 8.6438e-04 - val_mae: 0.0160 - learning_rate: 0.0010
Epoch 4/100
1753/1753 ━━━━━━━━━━━━━━━━━━━━ 82s 37ms/step - loss: 9.1098e-04 - mae: 0.0174 - val_loss: 8.3478e-04 - val_mae: 0.0169 - learning_rate: 0.0010
Epoch 5/100
1753/1753 ━━━━━━━━━━━━━━━━━━━━ 63s 36ms/step - loss: 8.7182e-04 - mae: 0.0171 - val_loss: 8.4826e-04 - val_mae: 0.0175 - learning_rate: 0.0010
Epoch 6/100
1753/1753 ━━━━━━━━━━━━━━━━━━━━ 81s 35ms/step - loss: 8.4475e-04 - mae: 0.0167 - val_loss: 8.8258e-04 - val_mae: 0.0183 - learning_rate: 0.0010
Epoch 7/100
1753/1753 ━━━━━━━━━━━━━━━━━━━━

# 6. EVALUATE MODEL

In [ ]:
print("\n[6/7] Evaluating model...")

# Predictions
y_train_pred = model.predict(X_train, verbose=0).flatten()
y_val_pred = model.predict(X_val, verbose=0).flatten()
y_test_pred = model.predict(X_test, verbose=0).flatten()

# Flatten y values to match prediction shapes
y_train = y_train.flatten()
y_val = y_val.flatten()
y_test = y_test.flatten()

# Denormalize test data for metrics reporting
y_test_denorm = scaler_target.inverse_transform(y_test.reshape(-1, 1)).flatten()
y_test_pred_denorm = scaler_target.inverse_transform(y_test_pred.reshape(-1, 1)).flatten()

# Metrics function
def calc_metrics(y_true, y_pred, set_name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    return {
        'Set': set_name,
        'MAE': mae,
        'RMSE': rmse,
        'R²': r2
    }

train_metrics = calc_metrics(y_train, y_train_pred, "Train")
val_metrics = calc_metrics(y_val, y_val_pred, "Validation")
test_metrics = calc_metrics(y_test_denorm, y_test_pred_denorm, "Test")

metrics_df = pd.DataFrame([train_metrics, val_metrics, test_metrics])

print("\n" + "="*80)
print("PERFORMANCE METRICS (Test: denormalized, Train/Val: normalized)")
print("="*80)
print(metrics_df.to_string(index=False))

# Save metrics
metrics_df.to_csv('Results/ErCNNLSTM_metrics_PM2.csv', index=False)
print("\nMetrics saved to: Results/eMaCNNLSTM_metrics.csv")

# PREDICTION INTERVALS (Narrow - 25th-75th percentile)

In [ ]:
print("\n" + "="*80)
print("PREDICTION INTERVALS (25th-75th Percentile)")
print("="*80)

# Calculate empirical percentiles from training residuals
train_residuals = y_train - y_train_pred
lower_percentile = np.percentile(train_residuals, 25)
upper_percentile = np.percentile(train_residuals, 75)

print(f"\nTraining residual 25th percentile: {lower_percentile:.6f}")
print(f"Training residual 75th percentile: {upper_percentile:.6f}")

# Apply intervals to test predictions
y_test_lower = y_test_pred + lower_percentile
y_test_upper = y_test_pred + upper_percentile

# Denormalize interval bounds
y_test_lower_denorm = scaler_target.inverse_transform(y_test_lower.reshape(-1, 1)).flatten()
y_test_upper_denorm = scaler_target.inverse_transform(y_test_upper.reshape(-1, 1)).flatten()

print(f"\nTest Predictions with IQR (25-75%):")
print(f"  Mean prediction: {np.mean(y_test_pred_denorm):.4f}")
print(f"  Mean interval: [{np.mean(y_test_lower_denorm):.4f}, {np.mean(y_test_upper_denorm):.4f}]")

# Save predictions with intervals
predictions_df = pd.DataFrame({
    'Actual': y_test_denorm,
    'Predicted': y_test_pred_denorm,
    'Lower_25Percentile': y_test_lower_denorm,
    'Upper_75Percentile': y_test_upper_denorm,
    'Residual': y_test_denorm - y_test_pred_denorm,
    'Abs_Error': np.abs(y_test_denorm - y_test_pred_denorm)
})
predictions_df.to_csv('Results/MidCNNLSTM_Predictions_PM2.csv', index=False)
print("Predictions with intervals saved to: Results/eMaCNNLSTM_Predictions.csv")

# 7. VISUALIZATIONS (Time Series, Quantile Analysis, SHAP)

In [ ]:
print("\nGenerating visualizations...")

from matplotlib import rcParams
rcParams['font.weight'] = 'bold'
plt.plot(y_test_denorm[0:240], color='blue', label = 'Observed')
plt.plot(y_test_pred_denorm[0:240], color='red', label = 'Predicted')
plt.ylabel('PM2.5', fontname="Times New Roman", size=20,fontweight="bold")
plt.xlabel('Time(Hrs)', fontname="Times New Roman", size=20,fontweight="bold")
plt.title('eMalahleni CNNLSTM Model', fontname="Times New Roman", size=28,fontweight="bold")
legend_properties = {'weight':'bold'}
plt.legend(prop=legend_properties)
#plt.savefig("Results/eMaCNNLSTMPred_PM2.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
errors = y_test_pred_denorm.flatten() - y_test_denorm.flatten()

# Calculate quantiles based on actual values
quantiles, bins = pd.qcut(y_test_denorm.flatten(), q=10, duplicates='drop', retbins=True)

# Calculate average error for each quantile
quantile_errors = []
for i in range(len(bins) - 1):
    group_indices = np.where((y_test_denorm.flatten() >= bins[i]) & (y_test_denorm.flatten() < bins[i+1]))[0]
    quantile_errors.append(errors[group_indices].mean())

# Round the bin edges for better readability
rounded_bins = np.round(bins, decimals=3)

# Plot quantiles vs. average errors
rcParams['font.weight'] = 'bold'
plt.figure(figsize=(8, 6))
plt.plot(range(1, len(quantiles.categories) + 1), quantile_errors, marker='o')
plt.xlabel('Quantile', fontname="Times New Roman", size=20, fontweight="bold")
plt.ylabel('Average Error', fontname="Times New Roman", size=20, fontweight="bold")
plt.title('eMalahleni CNNLSTM Model', fontname="Times New Roman", size=28, fontweight="bold")
plt.xticks(range(1, len(quantiles.categories) + 1), [f'{rounded_bins[i]:.3f} - {rounded_bins[i+1]:.3f}' for i in range(len(rounded_bins) - 1)], rotation=45)
plt.grid(True)
#plt.savefig("Results/eMaCNNLSTMMQuan_PM2.png", dpi=300, bbox_inches='tight')
plt.show()

# Save model
#model.save('multistation_cnn_lstm_best_model.h5')
print("Model saved to: multistation_cnn_lstm_best_model.h5")


# SHAP ANALYSIS FOR INTERPRETABILITY

In [7]:
if SHAP_AVAILABLE:
    print("\n" + "="*70)
    print("SHAP FEATURE IMPORTANCE ANALYSIS")
    print("="*70)
    
    try:
        def predict_wrapper(X_flat):
            """Convert flattened input back to 3D for model prediction."""
            if X_flat.ndim == 1:
                X_flat = X_flat.reshape(1, -1)
            n_samples = X_flat.shape[0]
            X_3d = X_flat.reshape(n_samples, X_train.shape[1], X_train.shape[2])
            return model.predict(X_3d, verbose=0).flatten()
        
        # Prepare sample data for SHAP (use first 100 test samples)
        n_shap_samples = min(100, len(X_test))
        X_shap_samples = X_test[:n_shap_samples].reshape(n_shap_samples, -1)  # Flatten to 2D
        X_background = X_shap_samples[:100]  # Use 100 background samples
        
        print("Initializing SHAP KernelExplainer (model-agnostic)...")
        explainer = shap.KernelExplainer(predict_wrapper, X_background)
        
        print("Computing SHAP values for test samples...")
        # Use first 100 test samples for explanation
        shap_values = explainer.shap_values(X_shap_samples[:100], nsamples=250)
        shap_values = np.array(shap_values).squeeze()  # Handle output shape
        
        # Create feature names for ALL 26 features across 24 timesteps
        # 26 features = 13 (eMalahleni) + 13 (Middelburg)
        stations_list = ['Middelburg', 'eMalahleni']
        em_features = list(pd.read_csv('MiddelburgIM.csv', sep=';', nrows=1, index_col=0).columns)
        feature_names = []
        for t in range(X_train.shape[1]):
            for station in stations_list:
                for feat in em_features:
                    feature_names.append(f"{station}_{feat}(t-{X_train.shape[1]-t})")
        
        # Ensure feature names match X_shap_samples shape
        if len(feature_names) != X_shap_samples.shape[1]:
            print(f"Warning: feature_names length {len(feature_names)} != X_shap_samples width {X_shap_samples.shape[1]}")
            feature_names = [f"f{i}" for i in range(X_shap_samples.shape[1])]
        
        # SHAP summary plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values, X_shap_samples[:100], 
                         feature_names=feature_names, show=False, max_display=20)
        plt.title('Middelburg GraphTransformer Model', fontweight='bold', size=14)
        plt.tight_layout()
        plt.savefig('Results/MidGraphTransformershap_PM2.png', dpi=300, bbox_inches='tight')
        print("SHAP summary plot saved to: Results/eMaCNNLSTMshap.png")
        plt.close()
        
        print("\u2705 SHAP analysis completed successfully!")
        
    except Exception as e:
        print(f"\u26a0\ufe0f SHAP analysis failed: {e}")
        print("Continuing without SHAP analysis...")
else:
    print("\n\u26a0\ufe0f SHAP not available. Install with: pip install shap")



SHAP FEATURE IMPORTANCE ANALYSIS
Initializing SHAP KernelExplainer (model-agnostic)...
Computing SHAP values for test samples...


  0%|          | 0/100 [00:00<?, ?it/s]

SHAP summary plot saved to: Results/eMaCNNLSTMshap.png
✅ SHAP analysis completed successfully!
